In [1]:
pip install sentence-transformers pandas numpy scikit-learn

In [ ]:
from sentence_transformers import SentenceTransformer

In [32]:
pip install faiss-gpu

  Using cached faiss_gpu-1.15.1-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 MB 8.0 MB/s eta 0:00:00


In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import csv
import pickle

# Modificar CSV

OASISBR dados de 2025 - 177.691 resultados

In [ ]:
csv.field_size_limit(150000)
df = pd.read_csv('data.csv', engine="python", on_bad_lines="warn").drop(
  columns=[
      'Ano de publicação',
      'ID Lattes dos orientadores',
      'ID Lattes dos membros da banca',
      'Título da instituição fonte',
      'Resumo + dados referenciais',
      'Título + primeiro autor + Palavras-chave',
      'Link do documento',
      'ID no Oasisbr',
      'Link OAI do registro',
      'Nome da fonte',
      'Acronimo da instituição no Oasisbr'
  ])

/tmp/ipykernel_3241/1772739304.py:2: ParserWarning: Skipping line 100858: unexpected end of data

  df = pd.read_csv('data.csv', engine="python", on_bad_lines="warn").drop(


In [ ]:
df['Resumo'] = df['Resumo'].str[:10000]
df['Resumo em Inglês'] = df['Resumo em Inglês'].str[:10000]

In [ ]:
df.to_csv('data_modified.csv', index=False)

In [ ]:
df.columns

Index(['Título do documento', 'Todos os autores', 'Orientadores',
       'Membros da banca', 'Sigla da instituição fonte',
       'Direito autoral e o acesso livre', 'Tipo de documento', 'Idioma',
       'Resumo', 'Dados referenciais do documento ',
       'Identificador persistente ARK', 'DOI do documento ', 'Palavras-chave ',
       'Resumo em Inglês', 'ISSN da revista ', 'ISBN do livro ', 'Referência'],
      dtype='object')

# Criar embeddings

In [6]:
df = pd.read_csv('data_modified.csv', engine="python")

In [7]:
def criar_texto_embedding(row):
    text = ""
    for column, value in row.items():
        text += f"{column}: {value}. "
    return text.strip()

In [8]:
df["texto_embedding"] = df.apply(
    criar_texto_embedding,
    axis=1
)

In [ ]:
modelo = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
embeddings = modelo.encode(
    df["texto_embedding"].tolist(),
    normalize_embeddings=True
)

print(embeddings.shape)

(100856, 384)


In [ ]:
np.save("embeddings.npy", embeddings)

In [6]:
embeddings = np.load("embeddings.npy")

In [33]:
import faiss

In [35]:
embeddings = embeddings.astype('float32')
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

queries = embeddings.copy()

similarity_threshold = 0.95

# lims: boundary pointers for each query
# D: similarity scores of the matches
# I: indices of the matching database items
lims, D, I = index.range_search(queries, similarity_threshold)

print(f"Total matches found above 95%: {len(I)}")

Total matches found above 95%: 149902


In [36]:
potential_duplicates = []

for query_idx in range(len(queries)):
    start_idx = lims[query_idx]
    end_idx = lims[query_idx + 1]

    scores = D[start_idx:end_idx]
    indices = I[start_idx:end_idx]

    for score, match_idx in zip(scores, indices):
        # Skip self-matches (same row)
        if query_idx == match_idx:
            continue

        # Optional: Avoid duplicate reverse pairs (e.g., keeping only A-B, not B-A)
        if query_idx < match_idx:
            potential_duplicates.append({
                'row_1': query_idx,
                'row_2': match_idx,
                'similarity': score
            })
potential_duplicates.sort(key=lambda x: x['similarity'], reverse=True)

print(f"Total potential duplicate pairs found: {len(potential_duplicates)}")

Total potential duplicate pairs found: 24523


In [37]:
with open("potential_duplicates.pkl", "wb") as f:
    pickle.dump(potential_duplicates, f)

In [3]:
with open("potential_duplicates.pkl", "rb") as f:
    potential_duplicates = pickle.load(f)

# Seleção por IA Generativa

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [4]:
model_name = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In [9]:
messages = []

for potential_duplicate in potential_duplicates:
        row_a = df.iloc[potential_duplicate['row_1']]['texto_embedding']
        row_b = df.iloc[potential_duplicate['row_2']]['texto_embedding']

        prompt = f"""
        You are an expert academic data curator. Compare the following two article entries from a database and determine if they represent the exact same work published under slightly different names or metadata variations, or if they are completely different papers.

        Similarity Score: {potential_duplicate['similarity']:.4f}

        --- ARTICLE A ---
        {row_a}

        --- ARTICLE B ---
        {row_b}

        Respond in the following format:
        Verdict: [DUPLICATE or DISTINCT]
        Reasoning: [Brief explanation of why]
        """

        messages.append([{"role": "user", "content": prompt}])

In [13]:
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

texts = tokenizer.apply_chat_template(
    messages[:5],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True
)

model_inputs = tokenizer(
    texts,
    return_tensors="pt",
    padding=True,
    truncation=True
).to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=1024
)

# trimmed_generated_ids = [
#     output_ids[len(input_ids):]
#     for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
# ]

decoded_outputs = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

for i, output in enumerate(decoded_outputs):
    print(f"--- Output {i+1} ---")
    print(output)

# text = tokenizer.apply_chat_template(
#     messages,
#     tokenize=False,
#     add_generation_prompt=True,
#     enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
# )
# model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# # conduct text completion
# generated_ids = model.generate(
#     **model_inputs,
#     max_new_tokens=32768
# )
# output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

# # parsing thinking content
# try:
#     # rindex finding 151668 (</think>)
#     index = len(output_ids) - output_ids[::-1].index(151668)
# except ValueError:
#     index = 0

# thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
# content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

# print("thinking content:", thinking_content)
# print("content:", content)

--- Output 1 ---
user

        You are an expert academic data curator. Compare the following two article entries from a database and determine if they represent the exact same work published under slightly different names or metadata variations, or if they are completely different papers.
        
        Similarity Score: 1.0000
        
        --- ARTICLE A ---
        Título do documento: Proposta de utilização de metodologias ágeis de gestão de projetos em empresas de base tecnológica residentes em habitats de inovação : um estudo de caso no Hub de Inovação Fronteira. Todos os autores: Oliveira, Rosana de Souza Costa de. Orientadores: Não informado pela instituição. Membros da banca: Não informado pela instituição. Sigla da instituição fonte: UEM. Direito autoral e o acesso livre: openAccess. Tipo de documento: masterThesis. Idioma: por. Resumo: Orientador: Prof. Dr. Marcelo Farid Pereira.. Dados referenciais do documento : reponame:Repositório Institucional da Universidade Estad